# 14 · Vector Index：为什么 HNSW 快

> 海量向量上逐条算距离太慢，于是有了 ANN（近似最近邻）索引。本课拆解主流索引，重点理解 **HNSW 为何快** 以及 **Recall/速度/内存** 三角权衡。

**本文件覆盖知识点**：Flat / IVF / HNSW / PQ / IVF-PQ / ANN / Exact vs Approximate Search / Trade-off(Recall·Speed·Memory)

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. Exact vs Approximate

- **Exact（精确）检索**：和每一个向量都比一遍（`IndexFlat`）。结果 100% 准，但 O(N)；
- **Approximate（近似）检索**：只扫描一部分候选。速度指数级提升，代价是可能漏掉真正最近邻（Recall<100%）。

实际应用几乎都用 ANN：**牺牲 1% 召回，换来 100 倍速度**，很划算。

## 2. 四大类索引一句话

| 索引 | 思路 | 特性 |
|------|------|------|
| **Flat** | 暴力全扫 | 精确、最慢，做基准 |
| **IVF** | 先聚类(K 个簇)，查询只进最近几簇 | 快，聚类质量影响召回 |
| **HNSW** | 分层小世界图：上层粗跳、下层细搜 | 速度/召回双优，主流默认 |
| **PQ** | 把向量压缩成短码，只算压缩距离 | 省内存，距离是近似 |
| **IVF-PQ** | IVF 分区 + PQ 压缩 | 亿级/内存受限场景 |


In [ ]:
# HNSW 为什么快：多跳“路由”而非逐层全查
print('''
HNSW = 多层的图导航

第3层(稀疏)  o---o---o        ← 从某节点开始
第2层       o-o-o-o-o-o      ← 快速跳向目标区域
第1层(稠密) o-o-o-o-o-o-o-o  ← 到最底层做细搜

搜索时从上往下: 顶层大步靠近 → 底层小步精确定位
复杂度 ~ O(log N) 量级，而非 O(N)。

代价: 多层图需要额外指针 → 内存比 Flat 高
      ef_search/ef_construction 调参影响 速度↔召回
''')

In [ ]:
# 在 FAISS 里对比 Flat(精确基准) 与 HNSW(近似) 的召回/耗时
import numpy as np, time, faiss
rng = np.random.default_rng(0)

N, D = 50_000, 64
xb = rng.random((N, D), dtype=np.float32)
xb /= np.linalg.norm(xb, axis=1, keepdims=True)          # 归一化
xq = rng.random((1000, D), dtype=np.float32); xq /= np.linalg.norm(xq, axis=1, keepdims=True)

# 1) Flat: 精确基准
flat = faiss.IndexFlatIP(D); flat.add(xb)
t0 = time.time(); Df, If = flat.search(xq, 10); t_flat = time.time() - t0

# 2) HNSW: 近似
hnsw = faiss.IndexHNSWFlat(D, 32)   # M=32（每层邻居数）
hnsw.add(xb)
t0 = time.time(); Dh, Ih = hnsw.search(xq, 10); t_hnsw = time.time() - t0

# 召回: HNSW 的 Top-10 与 Flat 的 Top-10 重合比例
recall = np.mean([len(set(Ih[i]) & set(If[i])) / 10 for i in range(len(xq))])
print(f'Flat 耗时: {t_flat*1000:.1f} ms | HNSW 耗时: {t_hnsw*1000:.1f} ms | 加速: {t_flat/t_hnsw:.1f}x')
print(f'HNSW vs 精确检索的 Top-10 召回率: {recall*100:.1f}%')

## 3. Trade-off：Recall ↔ Speed ↔ Memory

```text
Flat    : 召回100% · 最慢  · 原向量内存
IVF     : 召回高   · 快    · 聚类+原向量
HNSW    : 召回很高 · 很快  · 图指针多耗内存
PQ      : 召回中   · 很快  · 大幅省内存(压缩)
IVF-PQ  : 召回中上 · 极快  · 极省内存（亿级方案）
```

调 HNSW 的旋钮：`M`(层内邻居数,越大召回越高内存越多)、`efConstruction`(建图质量)、`efSearch`(查询范围，越大越准越慢)。



In [ ]:
# 知识点·真调说明：ANN 的工程权衡 —— 让模型解读“近似比精确快很多、召回仍接近 100%”意味着什么
_llm_live(
    prompt='前面用 FAISS 对比了 Flat(精确、逐条全扫) 和 HNSW(近似、分层图)：HNSW 明显更快，'
           '而 Top-10 召回仍接近精确检索的结果。请以检索工程师的身份解读两点：\n'
           '① 在 RAG 里，近似检索漏掉的那一点点召回，通常靠什么兜回来？\n'
           '② 举一种“近似检索不可接受”的业务场景，说明为什么那里连一点召回都不能丢。',
    system='你是资深检索工程师。要求分①②回答，每点不超过 2 句话，并给具体场景。',
    fallback='未配置 Key 的固定样例：\n'
             '① 近似漏掉的往往是“边界相似”的文档，而 RAG 下游通常还有候选池扩大 + rerank 精排（第 22 课）'
             '以及生成阶段的证据校验来兜底，所以少量漏召回不致命。\n'
             '② 例：风控黑名单或过敏史这类“必须命中唯一那条记录”的场景，漏一条就是事故，'
             '宁可慢也要精确检索或做强制二次校验。',
    temperature=0.2,
)
print('→ 别背“近似一定好”的结论：先定召回目标，再用第 34 课的检索指标验收你的索引。')

In [ ]:
# 知识点·真调说明：索引选型与调参 —— 把“Flat/IVF/HNSW/PQ/IVF-PQ 选谁、M/efSearch 拧多少”做一次真实咨询
_llm_live(
    prompt='真实约束：5000 万条、768 维、L2 归一化向量；单机 64GB 内存、不打算上分布式；'
           '要求 P95 延迟 < 30ms、Top-10 召回 ≥ 95%。\n'
           '请给出：① 主索引选 Flat / IVF / HNSW / PQ / IVF-PQ 中的哪一个并给一条核心理由（提示：先估算原始向量占多少内存）；'
           '② 若改用图索引路线，M 和 efSearch 两个旋钮分别主要影响哪组权衡，首轮大致取多少；'
           '③ 全量 Flat 为什么在这个规模下不可行。',
    system='你是向量索引性能顾问。要求分①②③回答，每点 1~2 句并尽量给数值，不写代码、不罗列无关产品。',
    fallback='未配置 Key 的固定样例：\n'
             '① 5000 万 × 768 维 × 4 字节 ≈ 154GB，单机 64GB 根本放不下原始向量：Flat、纯 HNSW 都被内存卡死，'
             '应选 IVF-PQ（IVF 先把检索范围收进几个近邻簇，PQ 把每向量压到几十字节）。\n'
             '② M 主要影响“召回↔内存”，efSearch 主要影响“召回↔速度”；首轮可取 M=32、efSearch=64~128，'
             '再用真实数据测召回后二分微调。\n'
             '③ Flat 每次查询要全扫并全量保存所有原始向量，此规模下内存与延迟都不达标，只适合当精确基准。',
    temperature=0.2,
)
print('→ 本课表格是“地图”，真实项目要在内存/延迟/召回约束里做数值决策——这是 RAG 检索层最容易被低估的一步。')

## 小结

- ANN 用近似换速度；**HNSW** 以分层图路由获得 O(logN) 级检索；
- 建索引要明确“召回目标”→ 选 Flat/IVF/HNSW/PQ 组合；
- 任何索引都要用**第 34 课的检索指标**验收。